# LeetCode 1962 — Remove Stones to Minimize the Total bằng Python

Link đề: https://leetcode.com/problems/remove-stones-to-minimize-the-total/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def minStoneSum(self, piles: List[int], k: int) -> int:
        ...
```

Notebook đi từ **thử mọi dãy thao tác** (vét cạn nhỏ, để kiểm chứng tham lam) → **mô phỏng quét tìm đống lớn nhất** → **max-heap** → **mảng đếm theo giá trị** (giá trị ≤ 10⁴) → **hai hàng đợi** (mảng đã sắp + hàng đợi giá trị đã chia đôi).

## 1. Mục tiêu học

1. Hiểu thao tác: chọn đống `x`, **bỏ đi** `⌊x/2⌋` viên ⇒ còn lại `x − ⌊x/2⌋ = ⌈x/2⌉` viên.
2. Thấy nhận xét tham lam: mỗi lần nên tác động lên **đống lớn nhất** (bỏ được nhiều nhất), và kiểm chứng bằng vét cạn.
3. Cài đặt **max-heap** trong Python bằng số âm và `heapreplace`.
4. Biết hai cách thay thế: **mảng đếm theo giá trị** và kỹ thuật **hai hàng đợi đơn điệu**.
5. Kiểm thử đối chiếu nhiều cách trên dữ liệu ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Cho mảng `piles` (`piles[i]` là số viên đá ở đống `i`) và số nguyên `k`. Thực hiện **đúng `k` lần** thao tác: chọn một đống `piles[i]` bất kỳ và **bỏ đi `⌊piles[i] / 2⌋`** viên. Một đống có thể được chọn nhiều lần.

Trả về **tổng số viên đá nhỏ nhất** còn lại sau `k` thao tác.

**Ví dụ 1:** `piles = [5,4,9]`, `k = 2` → `12`. Chia đống `9` → `[5,4,5]`; chia đống `5` → `[3,4,5]`; tổng `12`.

**Ví dụ 2:** `piles = [4,3,6,7]`, `k = 3` → `12`. `[4,3,3,7]` → `[4,3,3,4]` → `[2,3,3,4]`; tổng `12`.

**Ràng buộc:** `1 ≤ piles.length ≤ 10⁵`, `1 ≤ piles[i] ≤ 10⁴`, `1 ≤ k ≤ 10⁵`.

In [1]:
import heapq
import random
import time
from collections import deque
from functools import lru_cache
from typing import List

examples = [([5, 4, 9], 2, 12), ([4, 3, 6, 7], 3, 12)]

def op(x):
    """Một thao tác trên đống x: bỏ floor(x/2) viên."""
    return x - x // 2

print([(x, op(x)) for x in [1, 2, 3, 4, 5, 9, 10]], "<- (trước, sau)")

[(1, 1), (2, 1), (3, 2), (4, 2), (5, 3), (9, 5), (10, 5)] <- (trước, sau)


## 3. Kiến thức nền

### 3.1 Chia nguyên và làm tròn lên

`x // 2` là `⌊x/2⌋`. Phần còn lại `x − x//2` bằng `⌈x/2⌉` = `(x + 1) // 2`. Lưu ý đống `1` không giảm nữa (`1 − 0 = 1`).

In [2]:
for x in range(1, 8):
    print(x, x // 2, x - x // 2, (x + 1) // 2)

1 0 1 1
2 1 1 1
3 1 2 2
4 2 2 2
5 2 3 3
6 3 3 3
7 3 4 4


### 3.2 Max-heap bằng `heapq` với số âm

`heapq` là min-heap; lưu `-x` để `heap[0]` là `-max`. `heapreplace(h, v)` = lấy phần tử nhỏ nhất ra rồi đẩy `v` vào, nhanh hơn `heappop` + `heappush`.

In [3]:
h = [-x for x in [5, 4, 9]]
heapq.heapify(h)
print("lớn nhất:", -h[0])
top = -h[0]
heapq.heapreplace(h, -(top - top // 2))
print("sau một thao tác:", sorted((-x for x in h), reverse=True))

lớn nhất: 9
sau một thao tác: [5, 5, 4]


### 3.3 Mảng đếm theo giá trị

Khi giá trị nhỏ (`≤ 10⁴`), `cnt[v]` = số đống có đúng `v` viên. Đống lớn nhất là `v` lớn nhất có `cnt[v] > 0`.

In [4]:
cnt = [0] * 11
for x in [5, 4, 9]:
    cnt[x] += 1
print([(v, c) for v, c in enumerate(cnt) if c])

[(4, 1), (5, 1), (9, 1)]


### 3.4 `deque` làm hàng đợi

`deque.popleft()` và `append` đều `O(1)`. Ta dùng nó cho "hàng đợi các giá trị đã chia đôi".

In [5]:
q = deque([9, 5]); q.append(3)
print(q.popleft(), list(q))

9 [5, 3]


## 4. Ý tưởng nền tảng / trực giác

Thao tác trên đống `x` làm tổng **giảm đúng `⌊x/2⌋`**. Để tổng cuối nhỏ nhất ⇒ tổng lượng bỏ đi lớn nhất. Lượng bỏ `⌊x/2⌋` **tăng theo `x`** ⇒ trực giác: luôn chọn đống **lớn nhất hiện tại**.

Tính tay ví dụ 2: `[4,3,6,7]`, `k = 3`:
- Lớn nhất 7 ⇒ bỏ 3 ⇒ `[4,3,6,4]` (đề đi thứ tự khác nhưng tổng như nhau).
- Lớn nhất 6 ⇒ bỏ 3 ⇒ `[4,3,3,4]`.
- Lớn nhất 4 ⇒ bỏ 2 ⇒ `[2,3,3,4]` ⇒ tổng `12`.

Câu hỏi: chọn lớn nhất **mỗi bước** có chắc tối ưu cho cả `k` bước không? (Chia đôi đống lớn làm nó nhỏ đi, có thể ảnh hưởng tới lựa chọn về sau.) Mục 10 chứng minh; mục 5 kiểm bằng vét cạn.

In [6]:
def greedy_trace(piles, k):
    p = list(piles)
    for step in range(1, k + 1):
        i = max(range(len(p)), key=lambda t: p[t])
        before = p[i]; p[i] = op(p[i])
        print(f"  bước {step}: chọn {before} (bỏ {before // 2}) -> {p}, tổng {sum(p)}")
    return sum(p)

greedy_trace([4, 3, 6, 7], 3)

  bước 1: chọn 7 (bỏ 3) -> [4, 3, 6, 4], tổng 17
  bước 2: chọn 6 (bỏ 3) -> [4, 3, 3, 4], tổng 14
  bước 3: chọn 4 (bỏ 2) -> [2, 3, 3, 4], tổng 12


12

## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn mọi dãy thao tác (để kiểm chứng tham lam)

Thử mọi cách chọn đống cho từng bước — `nᵏ` dãy; dùng ghi nhớ trên **đa tập** các đống (sắp xếp thành tuple). Chỉ chạy được với `n, k` nhỏ.

In [7]:
def min_stone_exhaustive(piles, k):
    @lru_cache(maxsize=None)
    def go(state, rem):
        if rem == 0:
            return sum(state)
        best = None
        for i in range(len(state)):
            nxt = list(state); nxt[i] = op(nxt[i])
            v = go(tuple(sorted(nxt)), rem - 1)
            best = v if best is None else min(best, v)
        return best
    return go(tuple(sorted(piles)), k)

for piles_, k_, exp in examples:
    print(min_stone_exhaustive(piles_, k_), "(kỳ vọng", exp, ")")

12 (kỳ vọng 12 )
12 (kỳ vọng 12 )


### 5.2 Mô phỏng tham lam, quét tìm đống lớn nhất mỗi bước

In [8]:
def min_stone_scan(piles, k):
    p = list(piles)
    for _ in range(k):
        i = 0
        for t in range(1, len(p)):
            if p[t] > p[i]:
                i = t
        p[i] -= p[i] // 2
    return sum(p)

for piles_, k_, exp in examples:
    print(min_stone_scan(piles_, k_), "(kỳ vọng", exp, ")")

12 (kỳ vọng 12 )
12 (kỳ vọng 12 )


**Độ phức tạp:** 5.1 hàm mũ; 5.2 `O(k · n)` — với `k = n = 10⁵` là `10¹⁰`: quá chậm.

**Nút thắt:** mỗi bước quét lại toàn bộ để tìm max. Cấu trúc **heap** cho max trong `O(1)` và cập nhật trong `O(log n)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Tham lam:** ở mỗi bước, áp thao tác lên đống **lớn nhất** hiện tại.

**Quy tắc:** **nếu** có hai đống `x > y` **thì** áp lên `x` bỏ được `⌊x/2⌋ ≥ ⌊y/2⌋` — không bao giờ kém hơn **trong bước này**; và vì "lượng bỏ đi" của các bước sau chỉ phụ thuộc vào **đa tập giá trị**, ta chứng minh được tham lam tối ưu toàn cục (mục 10).

**Ba cách cài đặt nhanh:**
1. **Max-heap:** `O(n + k log n)`.
2. **Mảng đếm:** duyệt giá trị `v` từ `10⁴` xuống; mỗi đống giá trị `v` khi xử lý trở thành `⌈v/2⌉ < v` (khi `v ≥ 2`) — "rơi" xuống ô thấp hơn ⇒ `O(maxV + k)`.
3. **Hai hàng đợi:** các giá trị **mới sinh ra** (sau khi chia) xuất hiện theo thứ tự **không tăng** — vì ta luôn chia phần tử lớn nhất. Giữ `A` = các đống gốc đã sắp giảm, `Q` = hàng đợi giá trị mới; lớn nhất luôn ở đầu `A` hoặc đầu `Q` ⇒ `O(n log n + k)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Max-heap

In [9]:
def min_stone_heap(piles, k):
    h = [-x for x in piles]
    heapq.heapify(h)
    for _ in range(k):
        x = -h[0]
        heapq.heapreplace(h, -(x - x // 2))
    return -sum(h)

for piles_, k_, exp in examples:
    print(min_stone_heap(piles_, k_), "(kỳ vọng", exp, ")")

12 (kỳ vọng 12 )
12 (kỳ vọng 12 )


Lưu ý `-x // 2` và `-(x // 2)` **khác nhau** với số âm (`-9 // 2 = -5`). Code trên luôn đổi về số dương `x = -h[0]` trước khi chia để tránh nhầm (mục 13). **Độ phức tạp:** `O(n + k log n)`, bộ nhớ `O(n)`.

### 7.2 Cách 3 — Mảng đếm theo giá trị

In [10]:
def min_stone_bucket(piles, k):
    M = max(piles)
    cnt = [0] * (M + 1)
    for x in piles:
        cnt[x] += 1
    v = M
    while k > 0 and v > 1:
        if cnt[v] == 0:
            v -= 1
            continue
        use = min(cnt[v], k)            # chia 'use' đống giá trị v cùng lúc
        cnt[v] -= use
        cnt[v - v // 2] += use          # mỗi đống thành ceil(v/2) < v
        k -= use
    return sum(val * c for val, c in enumerate(cnt))

for piles_, k_, exp in examples:
    print(min_stone_bucket(piles_, k_), "(kỳ vọng", exp, ")")

12 (kỳ vọng 12 )
12 (kỳ vọng 12 )


Vì `⌈v/2⌉ < v` khi `v ≥ 2`, đống mới rơi xuống ô **thấp hơn** — con trỏ `v` không cần quay lại. Khi `v = 1` thì thao tác không đổi gì nữa (bỏ 0 viên) ⇒ dừng. **Độ phức tạp:** `O(n + M + số_lần_lặp)` với `M ≤ 10⁴` ⇒ `O(n + M)` (mỗi vòng hoặc giảm `v`, hoặc dùng hết một ô hay hết `k`). Bộ nhớ `O(M)`.

### 7.3 Cách 4 — Hai hàng đợi đơn điệu

In [11]:
def min_stone_two_queues(piles, k):
    A = sorted(piles, reverse=True)       # đống gốc, giảm dần
    Q = deque()                           # giá trị đã chia, được thêm theo thứ tự không tăng
    i = 0
    for _ in range(k):
        if Q and (i == len(A) or Q[0] >= A[i]):
            x = Q.popleft()
        else:
            x = A[i]; i += 1
        Q.append(x - x // 2)
    return sum(A[i:]) + sum(Q)

for piles_, k_, exp in examples:
    print(min_stone_two_queues(piles_, k_), "(kỳ vọng", exp, ")")

12 (kỳ vọng 12 )
12 (kỳ vọng 12 )


**Vì sao `Q` không tăng?** Ta luôn lấy ra **phần tử lớn nhất** `x` (đầu `A` hoặc đầu `Q`); các giá trị lấy ra theo thời gian là **không tăng**, nên `⌈x/2⌉` thêm vào `Q` cũng **không tăng** ⇒ đầu `Q` luôn là max của `Q`. **Độ phức tạp:** `O(n log n + k)` thời gian, `O(n + k)` bộ nhớ.

## 8. TRACE từng bước

Trace cách 2 (heap) trên ví dụ 1 `[5,4,9]`, `k = 2`:

| bước | heap (giảm dần) | lấy | bỏ | đẩy lại | tổng |
|---|---|---|---|---|---|
| 0 | 9, 5, 4 | | | | 18 |
| 1 | | 9 | 4 | 5 | 14 |
| 2 | 5, 5, 4 | 5 | 2 | 3 | 12 |

In [12]:
def trace_heap(piles, k):
    h = [-x for x in piles]; heapq.heapify(h)
    print(f"  ban đầu: {sorted((-x for x in h), reverse=True)}, tổng {-sum(h)}")
    for step in range(1, k + 1):
        x = -h[0]
        heapq.heapreplace(h, -(x - x // 2))
        print(f"  bước {step}: lấy {x}, bỏ {x // 2}, đẩy lại {x - x // 2} -> {sorted((-v for v in h), reverse=True)}, tổng {-sum(h)}")
    return -sum(h)

trace_heap([5, 4, 9], 2)
trace_heap([4, 3, 6, 7], 3)

  ban đầu: [9, 5, 4], tổng 18
  bước 1: lấy 9, bỏ 4, đẩy lại 5 -> [5, 5, 4], tổng 14
  bước 2: lấy 5, bỏ 2, đẩy lại 3 -> [5, 4, 3], tổng 12
  ban đầu: [7, 6, 4, 3], tổng 20
  bước 1: lấy 7, bỏ 3, đẩy lại 4 -> [6, 4, 4, 3], tổng 17
  bước 2: lấy 6, bỏ 3, đẩy lại 3 -> [4, 4, 3, 3], tổng 14
  bước 3: lấy 4, bỏ 2, đẩy lại 2 -> [4, 3, 3, 2], tổng 12


12

Trace cách 4 (hai hàng đợi) trên ví dụ 2:

In [13]:
def trace_two_queues(piles, k):
    A = sorted(piles, reverse=True); Q = deque(); i = 0
    for step in range(1, k + 1):
        if Q and (i == len(A) or Q[0] >= A[i]):
            x = Q.popleft(); src = "Q"
        else:
            x = A[i]; i += 1; src = "A"
        Q.append(x - x // 2)
        print(f"  bước {step}: lấy {x} từ {src}, thêm {x - x // 2} vào Q | A còn {A[i:]}, Q = {list(Q)}")
    total = sum(A[i:]) + sum(Q)
    print("  tổng =", total)
    return total

trace_two_queues([4, 3, 6, 7], 3)

  bước 1: lấy 7 từ A, thêm 4 vào Q | A còn [6, 4, 3], Q = [4]
  bước 2: lấy 6 từ A, thêm 3 vào Q | A còn [4, 3], Q = [4, 3]
  bước 3: lấy 4 từ Q, thêm 2 vào Q | A còn [4, 3], Q = [3, 2]
  tổng = 12


12

Trace cách 3 (mảng đếm) trên `piles = [9, 9, 9, 4]`, `k = 5` (ví dụ tự dựng, so với heap): để ý ba đống `9` được xử lý **cùng lúc** trong một vòng.

In [14]:
def trace_bucket(piles, k):
    M = max(piles); cnt = [0] * (M + 1)
    for x in piles: cnt[x] += 1
    v = M
    while k > 0 and v > 1:
        if cnt[v] == 0:
            v -= 1; continue
        use = min(cnt[v], k)
        cnt[v] -= use; cnt[v - v // 2] += use; k -= use
        print(f"  v={v}: chia {use} đống -> mỗi đống còn {v - v // 2}; còn k={k}; trạng thái = {[(val, c) for val, c in enumerate(cnt) if c]}")
    total = sum(val * c for val, c in enumerate(cnt))
    print("  tổng =", total)
    return total

assert trace_bucket([9, 9, 9, 4], 5) == min_stone_heap([9, 9, 9, 4], 5)

  v=9: chia 3 đống -> mỗi đống còn 5; còn k=2; trạng thái = [(4, 1), (5, 3)]
  v=5: chia 2 đống -> mỗi đống còn 3; còn k=0; trạng thái = [(3, 2), (4, 1), (5, 1)]
  tổng = 15


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def minStoneSum(self, piles: List[int], k: int) -> int:
        h = [-x for x in piles]                   # max-heap bằng số âm
        heapq.heapify(h)
        for _ in range(k):
            x = -h[0]
            if x == 1:                            # mọi đống đều là 1: không giảm được nữa
                break
            heapq.heapreplace(h, -(x - x // 2))
        return -sum(h)

for piles_, k_, exp in examples:
    assert Solution().minStoneSum(piles_, k_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi một **lời giải** là một dãy `k` lựa chọn đống. Tổng còn lại = tổng ban đầu − tổng lượng bỏ đi.

**Chuỗi lượng bỏ của một đống.** Nếu một đống ban đầu có `v` viên bị chọn `t` lần, lượng bỏ ở các lần lần lượt là `r₁ = ⌊v/2⌋`, `r₂ = ⌊⌈v/2⌉/2⌋`, … Dãy này **không tăng** (`r₂ ≤ r₁`, …) vì giá trị đống chỉ giảm và `⌊x/2⌋` tăng theo `x`. Lượng bỏ của mỗi đống chỉ phụ thuộc **số lần** nó được chọn, không phụ thuộc thứ tự xen kẽ với đống khác.

**Quy về chọn `k` phần tử lớn nhất.** Mỗi lời giải ứng với việc chọn, trên mỗi đống `i`, một **tiền tố** độ dài `tᵢ` của chuỗi lượng bỏ, với `Σ tᵢ = k`; tổng lượng bỏ là tổng các phần tử đã chọn. Vì mọi chuỗi đều không tăng, tập `k` phần tử **lớn nhất** trong hợp các chuỗi luôn có thể chọn sao cho trên mỗi chuỗi nó là một **tiền tố** (nếu lấy phần tử thứ `t + 1` mà bỏ phần tử thứ `t ≥` nó, ta đổi hai phần tử cho nhau mà tổng không giảm). Vậy tối ưu = tổng `k` lượng bỏ lớn nhất.

**Tham lam đạt đúng điều đó.** Ở mỗi bước, "đầu" của mỗi chuỗi là lượng bỏ kế tiếp của đống đó, và đống lớn nhất có đầu chuỗi lớn nhất. Tham lam lấy phần tử lớn nhất trong các đầu chuỗi — trong hợp các chuỗi không tăng, đó chính là phần tử lớn nhất còn lại ⇒ sau `k` bước được đúng `k` phần tử lớn nhất. Mục 12 kiểm chứng thêm bằng vét cạn trên hàng nghìn input nhỏ.

**Các cách cài đặt** (heap, mảng đếm, hai hàng đợi) đều mô phỏng **đúng** tham lam "chọn lớn nhất": heap trả max; mảng đếm xử lý từ giá trị lớn xuống và các giá trị mới luôn nhỏ hơn; hai hàng đợi giữ đầu mỗi hàng là max của hàng đó.

Minh họa lập luận "chọn `k` lượng bỏ lớn nhất": in chuỗi lượng bỏ của từng đống ở ví dụ 2, rồi lấy `k = 3` phần tử lớn nhất của hợp các chuỗi.

In [16]:
def removal_chain(v, steps=6):
    out = []
    for _ in range(steps):
        out.append(v // 2); v -= v // 2
    return out

piles_ = [4, 3, 6, 7]
chains = {x: removal_chain(x) for x in piles_}
for x, ch in chains.items():
    print(f"  đống {x}: lượng bỏ lần 1, 2, ... = {ch}")
top = sorted((r for ch in chains.values() for r in ch), reverse=True)[:3]
print("  3 lượng bỏ lớn nhất:", top, "-> tổng còn lại =", sum(piles_) - sum(top), "| Solution:", Solution().minStoneSum(piles_, 3))

  đống 4: lượng bỏ lần 1, 2, ... = [2, 1, 0, 0, 0, 0]
  đống 3: lượng bỏ lần 1, 2, ... = [1, 1, 0, 0, 0, 0]
  đống 6: lượng bỏ lần 1, 2, ... = [3, 1, 1, 0, 0, 0]
  đống 7: lượng bỏ lần 1, 2, ... = [3, 2, 1, 0, 0, 0]
  3 lượng bỏ lớn nhất: [3, 3, 2] -> tổng còn lại = 12 | Solution: 12


## 11. Phân tích Time / Space Complexity

Gọi `n = len(piles)`, `M = max(piles) ≤ 10⁴`.

| Cách | Time | Space |
|---|---|---|
| 1a. Vét cạn | hàm mũ | hàm mũ |
| 1b. Quét max | `O(k · n)` | `O(n)` |
| 2. Max-heap | `O(n + k log n)` | `O(n)` |
| 3. Mảng đếm | `O(n + M)` | `O(M)` |
| 4. Hai hàng đợi | `O(n log n + k)` | `O(n + k)` |

In [17]:
rnd = random.Random(1962)
big = [rnd.randint(1, 10**4) for _ in range(10**5)]
for name, fn in [("heap", min_stone_heap), ("bucket", min_stone_bucket), ("two_queues", min_stone_two_queues), ("Solution", Solution().minStoneSum)]:
    t0 = time.perf_counter(); r = fn(big, 10**5); dt = time.perf_counter() - t0
    print(f"{name:<11}: {r:>10}  {dt * 1000:7.1f} ms")
t0 = time.perf_counter(); r = min_stone_scan(big[:2000], 2000); dt = time.perf_counter() - t0
print(f"scan n=k=2000: {r} ({dt * 1000:.0f} ms) | heap: {min_stone_heap(big[:2000], 2000)}")

heap       :  208386530     67.7 ms
bucket     :  208386530      4.5 ms


two_queues :  208386530     28.3 ms


Solution   :  208386530     68.6 ms


scan n=k=2000: 4387234 (89 ms) | heap: 4387234


## 12. Bộ test

In [18]:
ALL = {
    "scan": min_stone_scan,
    "heap": min_stone_heap,
    "bucket": min_stone_bucket,
    "two_queues": min_stone_two_queues,
    "Solution": lambda p, k: Solution().minStoneSum(list(p), k),
}

def check(piles, k, exp=None, exhaustive=True):
    ref = min_stone_exhaustive(piles, k) if exhaustive else min_stone_heap(piles, k)
    for name, fn in ALL.items():
        if not exhaustive and name == "scan":
            continue
        got = fn(list(piles), k)
        assert got == ref, (name, piles, k, got, ref)
    if exp is not None:
        assert ref == exp
    return ref

for piles_, k_, exp in examples:
    check(piles_, k_, exp)
print("Ví dụ đề: OK")

check([1], 5, 1)                    # đống 1 không giảm được
check([1, 1, 1], 3, 3)
check([10000], 1, 5000)
check([2], 1, 1)
check([3, 3], 1, 5)                 # 3 -> 2
check([7, 1, 1], 10, 3)             # 7 -> 4 -> 2 -> 1, rồi không đổi
print("Edge case: OK")

rnd = random.Random(2008)
for _ in range(1500):
    n = rnd.randint(1, 4)
    piles_ = [rnd.randint(1, 30) for _ in range(n)]
    check(piles_, rnd.randint(1, 5))
print("1500 bộ nhỏ so với vét cạn mọi dãy thao tác: OK")

for _ in range(30):
    n = rnd.randint(1, 3000)
    piles_ = [rnd.randint(1, 10**4) for _ in range(n)]
    check(piles_, rnd.randint(1, 10**5), exhaustive=False)
print("30 bộ cỡ lớn so giữa các cách nhanh: OK")

Ví dụ đề: OK
Edge case: OK


1500 bộ nhỏ so với vét cạn mọi dãy thao tác: OK


30 bộ cỡ lớn so giữa các cách nhanh: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nhầm "bỏ đi `⌊x/2⌋`" thành "còn lại `⌊x/2⌋`"

In [19]:
def min_stone_bug_floor(piles, k):
    h = [-x for x in piles]; heapq.heapify(h)
    for _ in range(k):
        x = -h[0]
        heapq.heapreplace(h, -(x // 2))           # SAI: còn lại phải là x - x//2 = ceil(x/2)
    return -sum(h)

piles_, k_ = [5, 4, 9], 2
print("sai :", min_stone_bug_floor(piles_, k_), "| đúng:", Solution().minStoneSum(piles_, k_))

sai : 10 | đúng: 12


**Vì sao sai:** với `x` lẻ, `x // 2` nhỏ hơn số đá thực còn lại (`9` còn `5`, không phải `4`). **Sửa:** `x - x // 2`.

### Lỗi 2 — Dùng min-heap (quên đổi dấu) ⇒ chia đống nhỏ nhất

In [20]:
def min_stone_bug_minheap(piles, k):
    h = list(piles); heapq.heapify(h)              # SAI: heap[0] là đống NHỎ nhất
    for _ in range(k):
        x = h[0]
        heapq.heapreplace(h, x - x // 2)
    return sum(h)

piles_, k_ = [5, 4, 9], 2
print("sai :", min_stone_bug_minheap(piles_, k_), "| đúng:", Solution().minStoneSum(piles_, k_))

sai : 15 | đúng: 12


**Sửa:** lưu `-x` (max-heap) như code nộp.

### Lỗi 3 — Chia trên số âm: `-x // 2` thay vì `-(x // 2)`

In [21]:
def min_stone_bug_negdiv(piles, k):
    h = [-x for x in piles]; heapq.heapify(h)
    for _ in range(k):
        neg = h[0]                                 # neg = -x
        heapq.heapreplace(h, neg - neg // 2)       # SAI: neg // 2 làm tròn về -vô cực
    return -sum(h)

for piles_, k_ in [([5, 4, 9], 2), ([3], 1)]:
    print(f"{piles_}, k={k_}: sai = {min_stone_bug_negdiv(piles_, k_)}, đúng = {Solution().minStoneSum(piles_, k_)}")

[5, 4, 9], k=2: sai = 10, đúng = 12
[3], k=1: sai = 1, đúng = 2


**Vì sao sai:** `-9 // 2 = -5` (làm tròn xuống), nên `neg − neg // 2 = -9 + 5 = -4` ⇒ đống thành `4` thay vì `5`. **Sửa:** đổi về số dương trước: `x = -h[0]`, rồi `-(x - x // 2)`.

### Lỗi 4 — Sắp xếp một lần rồi chia `k` đống đầu tiên

In [22]:
def min_stone_bug_sort_once(piles, k):
    p = sorted(piles, reverse=True)
    for i in range(min(k, len(p))):               # SAI: mỗi đống chỉ chia một lần, không sắp lại
        p[i] -= p[i] // 2
    return sum(p)

for piles_, k_ in [([9, 1], 2), ([100, 50], 3)]:
    print(f"{piles_}, k={k_}: sai = {min_stone_bug_sort_once(piles_, k_)}, đúng = {Solution().minStoneSum(piles_, k_)}")

[9, 1], k=2: sai = 6, đúng = 4
[100, 50], k=3: sai = 75, đúng = 50


**Vì sao sai:** sau khi chia, một đống có thể **vẫn lớn nhất** (`9 → 5`, vẫn lớn hơn `1`) và nên được chia tiếp. Danh sách phải được "sắp lại" sau mỗi thao tác — chính là việc heap làm. **Sửa:** heap / mảng đếm / hai hàng đợi.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Thực hiện `k` lần, mỗi lần chọn phần tử lớn nhất/nhỏ nhất rồi biến đổi" ⇒ **heap**.
- Python không có max-heap ⇒ lưu số âm, **cẩn thận phép chia với số âm**.
- Giá trị nhỏ (`≤ 10⁴`) ⇒ có thể thay heap bằng **mảng đếm**.
- Nếu giá trị sinh ra **đơn điệu** theo thời gian ⇒ **hai hàng đợi** thay cho heap (giống bài "Ugly Number", "Merge k sorted").
- Dừng sớm khi max đã là `1` (thao tác không còn tác dụng) — tiết kiệm thời gian khi `k` rất lớn.

## 15. Biến thể của bài

1. **LeetCode 2558 — Take Gifts From the Richest Pile:** thay `⌈x/2⌉` bằng `⌊√x⌋` — cùng khung heap.
2. **LeetCode 2530 — Maximal Score After Applying K Operations:** cộng điểm `x` rồi thay bằng `⌈x/3⌉` — heap tối đa tổng.
3. **LeetCode 1046 — Last Stone Weight:** lấy hai viên lớn nhất mỗi lần.
4. **Chi phí khác nhau mỗi đống** (bỏ `⌊x/2⌋` tốn `c_i`): cần ưu tiên theo "lợi ích/chi phí" — tham lam có thể không còn đúng.
5. **Tối thiểu số thao tác để tổng ≤ T:** lặp tham lam tới khi tổng ≤ T (LeetCode 2208 — Minimum Operations to Halve Array Sum, với số thực).

In [23]:
import math

def pick_gifts(gifts, k):
    """LeetCode 2558."""
    h = [-x for x in gifts]; heapq.heapify(h)
    for _ in range(k):
        x = -h[0]
        heapq.heapreplace(h, -math.isqrt(x))
    return -sum(h)

def max_kelements(nums, k):
    """LeetCode 2530."""
    h = [-x for x in nums]; heapq.heapify(h); score = 0
    for _ in range(k):
        x = -h[0]; score += x
        heapq.heapreplace(h, -((x + 2) // 3))
    return score

print("2558 [25,64,9,4,100], k=4 ->", pick_gifts([25, 64, 9, 4, 100], 4))
print("2530 [1,10,3,3,3], k=3 ->", max_kelements([1, 10, 3, 3, 3], 3))

2558 [25,64,9,4,100], k=4 -> 29
2530 [1,10,3,3,3], k=3 -> 17


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Vét cạn | mọi dãy thao tác | hàm mũ | hàm mũ | kiểm chứng tham lam |
| 1b. Quét max | tham lam, tìm max tuyến tính | `O(kn)` | `O(n)` | hiểu bài |
| 2. Max-heap | tham lam + heap | `O(n + k log n)` | `O(n)` | **nộp bài**, tổng quát |
| 3. Mảng đếm | giá trị nhỏ, rơi xuống ô thấp | `O(n + M)` | `O(M)` | `M` nhỏ, nhanh nhất |
| 4. Hai hàng đợi | giá trị mới đơn điệu | `O(n log n + k)` | `O(n + k)` | khi không có heap / học kỹ thuật |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minStoneSum([10], 3)` và `([1, 1, 100], 2)`?

**Bài 2 (tính tay):** Trace max-heap cho `piles = [8, 8, 3]`, `k = 4` (ghi đống được chọn và tổng sau mỗi bước).

**Bài 3 (điền TODO):** Hoàn thiện `min_ops_to_reach(piles, T)`: số thao tác **ít nhất** để tổng `≤ T` (trả `−1` nếu không thể).

**Bài 4 (tự viết lại):** Viết lại cách 3 (mảng đếm) nhưng trả về luôn **mảng các đống cuối cùng** (đã sắp giảm dần).

**Bài 5 (mini challenge):** Biến thể "mỗi thao tác bỏ `⌊x/3⌋` viên". Viết `min_stone_sum_third(piles, k)` và kiểm bằng vét cạn rằng tham lam vẫn đúng.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def min_ops_to_reach(piles, T):
    # TODO: heap; lặp chia max tới khi tổng <= T; nếu max == 1 mà tổng > T thì -1
    pass

# Bài 4 — TODO
def final_piles_bucket(piles, k):
    # TODO: như min_stone_bucket, cuối cùng dựng lại danh sách từ cnt
    pass

# Bài 5 — TODO
def min_stone_sum_third(piles, k):
    # TODO: heap với x -> x - x // 3
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `10 → 5 → 3 → 2`.
- Mức 2: `[1, 1, 100]`: chia `100` hai lần.
- Mức 3: `2` và `1 + 1 + 25 = 27`.

**Bài 2**
- Mức 1: Chọn một trong hai đống 8.
- Mức 2: `[4, 8, 3]` → chọn 8 → `[4, 4, 3]`.
- Mức 3: Tiếp chọn 4, rồi 4: `[2, 2, 3]`, tổng 7.

**Bài 3**
- Mức 1: Tham lam vẫn tối ưu (tối đa lượng bỏ mỗi bước).
- Mức 2: Dừng khi tổng ≤ T.
- Mức 3: Tổng không thể nhỏ hơn số đống (mỗi đống ≥ 1).

**Bài 4**
- Mức 1: Duyệt `cnt` từ lớn xuống.
- Mức 2: `[v] * cnt[v]`.
- Mức 3: So với heap đã sắp.

**Bài 5**
- Mức 1: Còn lại `x − ⌊x/3⌋`.
- Mức 2: Lượng bỏ `⌊x/3⌋` vẫn tăng theo `x`.
- Mức 3: Kiểm bằng phiên bản vét cạn với `op` thay đổi.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
r1 = (Solution().minStoneSum([10], 3), Solution().minStoneSum([1, 1, 100], 2))
print("Bài 1:", r1)
assert r1 == (2, 27)

# Bài 2
assert trace_heap([8, 8, 3], 4) == 7

# Bài 3
def min_ops_to_reach_sol(piles, T):
    h = [-x for x in piles]; heapq.heapify(h)
    total = sum(piles); ops = 0
    while total > T:
        x = -h[0]
        if x == 1:
            return -1
        heapq.heapreplace(h, -(x - x // 2))
        total -= x // 2; ops += 1
    return ops

def min_ops_to_reach_brute(piles, T, limit=8):
    for k in range(0, limit + 1):
        if min_stone_exhaustive(piles, k) <= T if k else sum(piles) <= T:
            return k
    return None

rnd = random.Random(3)
for _ in range(400):
    p = [rnd.randint(1, 20) for _ in range(rnd.randint(1, 3))]
    T = rnd.randint(len(p), sum(p))
    b = min_ops_to_reach_brute(p, T)
    if b is not None:
        assert min_ops_to_reach_sol(p, T) == b
print("Bài 3 OK | [5,4,9], T=12 ->", min_ops_to_reach_sol([5, 4, 9], 12), "| [1,1], T=1 ->", min_ops_to_reach_sol([1, 1], 1))

Bài 1: (2, 27)
  ban đầu: [8, 8, 3], tổng 19
  bước 1: lấy 8, bỏ 4, đẩy lại 4 -> [8, 4, 3], tổng 15
  bước 2: lấy 8, bỏ 4, đẩy lại 4 -> [4, 4, 3], tổng 11
  bước 3: lấy 4, bỏ 2, đẩy lại 2 -> [4, 3, 2], tổng 9
  bước 4: lấy 4, bỏ 2, đẩy lại 2 -> [3, 2, 2], tổng 7
Bài 3 OK | [5,4,9], T=12 -> 2 | [1,1], T=1 -> -1


In [26]:
# Bài 4
def final_piles_bucket_sol(piles, k):
    M = max(piles); cnt = [0] * (M + 1)
    for x in piles: cnt[x] += 1
    v = M
    while k > 0 and v > 1:
        if cnt[v] == 0:
            v -= 1; continue
        use = min(cnt[v], k)
        cnt[v] -= use; cnt[v - v // 2] += use; k -= use
    out = []
    for val in range(M, 0, -1):
        out += [val] * cnt[val]
    return out

def final_piles_heap(piles, k):
    h = [-x for x in piles]; heapq.heapify(h)
    for _ in range(k):
        x = -h[0]; heapq.heapreplace(h, -(x - x // 2))
    return sorted((-x for x in h), reverse=True)

rnd = random.Random(4)
for _ in range(500):
    p = [rnd.randint(1, 200) for _ in range(rnd.randint(1, 30))]
    kk = rnd.randint(1, 100)
    assert final_piles_bucket_sol(p, kk) == final_piles_heap(p, kk)
print("Bài 4 OK | ví dụ 2:", final_piles_bucket_sol([4, 3, 6, 7], 3))

# Bài 5
def min_stone_sum_third_sol(piles, k):
    h = [-x for x in piles]; heapq.heapify(h)
    for _ in range(k):
        x = -h[0]; heapq.heapreplace(h, -(x - x // 3))
    return -sum(h)

def exhaustive_third(piles, k):
    @lru_cache(maxsize=None)
    def go(state, rem):
        if rem == 0: return sum(state)
        return min(go(tuple(sorted(state[:i] + (state[i] - state[i] // 3,) + state[i + 1:])), rem - 1) for i in range(len(state)))
    return go(tuple(sorted(piles)), k)

rnd = random.Random(5)
for _ in range(800):
    p = [rnd.randint(1, 40) for _ in range(rnd.randint(1, 4))]
    kk = rnd.randint(1, 5)
    assert min_stone_sum_third_sol(p, kk) == exhaustive_third(p, kk), (p, kk)
print("Bài 5 OK | [5,4,9], k=2 ->", min_stone_sum_third_sol([5, 4, 9], 2))

Bài 4 OK | ví dụ 2: [4, 3, 3, 2]
Bài 5 OK | [5,4,9], k=2 -> 13


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "k lần lấy max rồi biến đổi" (max-heap Python):**

```python
h = [-x for x in arr]
heapq.heapify(h)
for _ in range(k):
    x = -h[0]                     # đổi về số dương TRƯỚC khi tính toán
    heapq.heapreplace(h, -f(x))   # f(x) = x - x // 2 ở bài này
return -sum(h)
```

**Checklist:**
- [ ] Còn lại `x − x//2` (không phải `x//2`)?
- [ ] Heap là **max-heap** (lưu số âm)?
- [ ] Không chia trực tiếp trên số âm?
- [ ] Sau mỗi thao tác, phần tử được "sắp lại" (heap), không chỉ sắp một lần?
- [ ] Dừng sớm khi max là 1 (tùy chọn)?
- [ ] Đã kiểm tham lam bằng vét cạn trên input nhỏ?
- [ ] Chữ ký đúng: `def minStoneSum(self, piles: List[int], k: int) -> int`?